In [1]:
import cv2
import numpy as np 
import matplotlib.pyplot as plt
from utils.yolo import *
from utils.video import *
from utils.color import *
from utils.ROI import * 
import os
import json
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
in_path  = "data/splits_fixed/trick_2.mp4"
out_path = "output/trick_2_output001.mp4"

In [20]:
cap = cv2.VideoCapture(in_path)
if not cap.isOpened():
    raise IOError(f"Could not open {in_path}")
fps = cap.get(cv2.CAP_PROP_FPS)
w   = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
fourcc = cv2.VideoWriter_fourcc(*"mp4v") 
writer = cv2.VideoWriter(out_path, fourcc, fps, (w, h))
fun(cap, writer)
cap.release()
writer.release()
print("Saved:", out_path)

Saved: output/trick_2_ball_change_color_static.mp4


In [19]:
in_path  = "data/splits_fixed/trick_2.mp4"
out_path = "output/trick_2_ball_change_color_static.mp4"
def fun(cap: cv2.VideoCapture, writer: cv2.VideoWriter):

    switch = 0              # 0=no event yet, 1=first event, 2=second event...
    overlap_prev = False    # was overlapping in previous frame?

    while True:
        ok, frame = cap.read()
        output = frame
        if not ok:
            break

        # Compute masks
        mask_blue = detect_color(frame, [0,0,255], [100,100],[255,255], tuning=25)
        mask_red  = detect_color(frame, [255,0,5], [55,55],[255,255], tuning=25)
        roi_blue = roi(mask_blue, 10, 20)
        roi_red  = roi(mask_red, 3, 20)

        overlap = cv2.bitwise_and(roi_blue, roi_red)
        overlap_pixels = np.count_nonzero(overlap)

        # ----- STABLE EVENT DETECTION -----
        overlap_now = overlap_pixels > 30   # threshold

        if overlap_now and not overlap_prev:
            # A NEW overlap event occurred here
            switch += 1

        overlap_prev = overlap_now

        # ----- STATE LOGIC -----
        state = switch >= 1    # state becomes True after first switch
        
        
        #--------Change COLOR-------
        mask_blue = roi(mask_blue, 10, 15)
        if switch == 1:
            output = change_color_mask(frame, [0,0,255],[255,0,0] , mask_blue)
        if switch ==2 :
            output = change_color_mask(frame, [0,0,255],[0,255,0] , mask_blue)
        
        # ----- Debug -----
        cv2.putText(output, f"Overlap: {overlap_pixels}",
                    (50,50), cv2.FONT_HERSHEY_SIMPLEX, 1,(0,255,255),2)
        cv2.putText(output, f"Switch: {switch}",
                    (50,90), cv2.FONT_HERSHEY_SIMPLEX, 1,(0,255,0),2)
        cv2.putText(output,f"State: {state}",
                    (50,130), cv2.FONT_HERSHEY_SIMPLEX, 1,(0,255,0),2)

        writer.write(output)

    return 1


In [ ]:
1%2
if 1 :
  print("ok")

ok
